## Status (created 2026-08-04) -- READ BEFORE RUNNING

**No training. No GPU needed.** This notebook only reads per-batch CSVs that the
ablation runs already wrote to disk, and runs a statistical test on them.

Reviewer nG29 (round 3): *"several arms show differences with overlapping standard
deviations, such as L_nassoc on purity (.972+-.09 -> .962+-.09) and the stop-gradient on
modularity (.615+-.08 -> .555+-.07)."*

**Why overlapping std is not the right test here.** The `+-` in those numbers is the
spread of modularity **across the batches of one dataset** -- it says pancreas batch 3 is
harder than pancreas batch 7. It is not the uncertainty of the full-vs-ablated
comparison. The same physical batches appear in the full-model run and in every ablated
run, so the two arrays are **paired**: `full[i]` and `ablated[i]` are the same batch.
If every batch drops by ~0.06, that is a completely consistent effect even though the
across-batch spread is 0.08. A paired test sees this; comparing error bars cannot.

This is the same reasoning -- and the same test -- already used for the Harmony/ComBat
baselines (`p=.035* (7/8)`), so the ablation and the baseline tables end up internally
consistent. The codebase already states it explicitly, in
`graph_batch_significance_paired`'s docstring:

> *Only 'modularity_per_batch' is pairable: batch is a shared, method-independent unit.*

**What this notebook does NOT answer.** It answers *"is the difference real given
batch-to-batch spread?"* It does **not** answer *"would the difference replicate from a
different random initialisation?"* Only seeds answer that -- see
`ablation_seed_variance.ipynb`.

**Which metrics are testable this way**

| metric | `+-` is across | pairable? |
|---|---|---|
| `modularity_per_batch` | batches | yes |
| rare coverage / homogeneity / F1 | batches | yes |
| `purity_per_mc` | metacells | **no** -- different runs produce different metacells, nothing to pair |

So the stop-gradient/modularity claim is testable here. The `L_nassoc`/purity claim is
**not** -- purity's spread is per-metacell. That claim should be re-anchored to rare-cell
homogeneity (per batch, and it drops on all three datasets), which this notebook does
test.

Everything is written to `OUT_DIR` after each dataset, so an interrupted session never
loses finished work.

## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 195.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.9/155.9 kB 21.8 MB/s eta 0:00:00


  Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=

In [1]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [2]:
import os, re, json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from interpretable_ssl.configs.paths import get_dataset_model_dir

print("ready")

ready


## Config

`ABLATION_PREFIX` is the `experiment_prefix` that `run_ablation_variant()` used
(`tasks.py:423`), which becomes the literal prefix of every ablation run folder:
`{prefix}_{tag}_{model-name tokens}`.

Datasets are ordered pancreas-first deliberately -- pancreas is the only dataset the
reviewer quoted numbers from, so if the session dies partway you still have the result
that matters most.

In [3]:
DATASETS_ORDER = ['pancreas', 'lung', 'pbmc-immune']   # pancreas first: the one nG29 quoted
DATASET_DISPLAY = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

ABLATION_PREFIX = 'ablation'   # experiment_prefix passed to run_ablation_variant()

# Tag of the un-ablated reference arm. None = auto-detect (looks for 'full', 'full_model',
# 'base', 'baseline', 'none'). Set explicitly if auto-detection picks the wrong one.
REF_TAG = None

OUT_DIR = '/content/drive/MyDrive/rebuttal_results/ablation_paired/'
os.makedirs(OUT_DIR, exist_ok=True)

SKIP_IF_EXISTS = True   # reload a dataset's saved result instead of recomputing

print('OUT_DIR:', OUT_DIR)

OUT_DIR: /content/drive/MyDrive/rebuttal_results/ablation_paired/


## Step 1 -- discover the ablation runs on disk

The notebook that defined the ablation tags (`component_ablation.ipynb`) was overwritten
by the ComBat notebook, so the tag list is not hardcoded here -- it is recovered from the
folder names instead.

A run folder looks like `ablation_{tag}_ds-panc_NP220_prtInit-wayp_...`. The tag is
everything between the prefix and the first *model-name* token, which the trainer's
naming always draws from a fixed vocabulary (`ds-`, `NP<n>`, `prtInit-`, `aff-`, ...).

**Check the printed table before continuing.** If a tag looks wrong, fill in
`MANUAL_TAGS` below and re-run.

In [4]:
# Tokens the trainer's model-name builder emits -- the tag ends at the first of these.
_NAME_TOKEN = re.compile(
    r'^(ds-|NP\d|prtInit-|aff-|bs\d|cvae|lprec|usim-|lpu|pum-|lna\d|nagg-|upm-|e\d|ecal\d|v\d+$)'
)

MANUAL_TAGS = {}   # optional override: {folder_name: tag}


def _tag_from_folder(folder, prefix):
    if folder in MANUAL_TAGS:
        return MANUAL_TAGS[folder]
    rest = folder[len(prefix) + 1:]
    parts, out = rest.split('_'), []
    for p in parts:
        if _NAME_TOKEN.match(p):
            break
        out.append(p)
    return '_'.join(out) if out else rest


def discover_ablation_runs(ds_id, prefix=ABLATION_PREFIX):
    """{tag: run_dir} for every `{prefix}_*` folder in this dataset's model dir."""
    base = get_dataset_model_dir(ds_id)
    found = {}
    if not os.path.isdir(base):
        print(f'  [{ds_id}] model dir does not exist: {base}')
        return found
    for e in sorted(os.listdir(base)):
        full = os.path.join(base, e)
        if not os.path.isdir(full) or not e.startswith(prefix + '_'):
            continue
        tag = _tag_from_folder(e, prefix)
        if tag in found:
            print(f'  [{ds_id}] WARNING: tag {tag!r} matched twice '
                  f'({os.path.basename(found[tag])} and {e}) -- keeping the later one')
        found[tag] = full
    return found


rows = []
runs_by_ds = {}
for ds in DATASETS_ORDER:
    runs = discover_ablation_runs(ds)
    runs_by_ds[ds] = runs
    for tag, d in runs.items():
        rows.append({
            'dataset': ds, 'tag': tag, 'folder': os.path.basename(d),
            'has_modularity_per_batch': os.path.exists(os.path.join(d, 'modularity_per_batch.csv')),
            'has_umap_cells': os.path.exists(os.path.join(d, 'umap_cells.csv')),
        })

df_runs = pd.DataFrame(rows)
print(f'\nfound {len(df_runs)} ablation run folder(s) across {len(DATASETS_ORDER)} dataset(s)')
df_runs


found 24 ablation run folder(s) across 3 dataset(s)


,dataset,tag,folder,has_modularity_per_batch,has_umap_cells
0,pancreas,fixed_temp,ablation_fixed_temp_ds-panc_NP220_prtInit-wayp...,True,True
1,pancreas,full,ablation_full_ds-panc_NP220_prtInit-wayp_aff-a...,True,True
2,pancreas,kmeans_init,ablation_kmeans_init_ds-panc_NP220_aff-arbf_lp...,True,True
3,pancreas,no_community,ablation_no_community_ds-panc_NP220_prtInit-wa...,True,True
4,pancreas,no_nassoc,ablation_no_nassoc_ds-panc_NP220_prtInit-wayp_...,True,True
5,pancreas,no_recon,ablation_no_recon_ds-panc_NP220_prtInit-wayp_a...,True,True
6,pancreas,no_usage,ablation_no_usage_ds-panc_NP220_prtInit-wayp_a...,True,True
7,pancreas,stopgrad_off,ablation_stopgrad_off_ds-panc_NP220_prtInit-wa...,True,True
8,lung,fixed_temp,ablation_fixed_temp_ds-lung_prtInit-wayp_aff-a...,True,True
9,lung,full,ablation_full_ds-lung_prtInit-wayp_aff-arbf_lp...,True,True


In [5]:
# Resolve the reference (un-ablated) arm.
_REF_CANDIDATES = ['full', 'full_model', 'fullmodel', 'base', 'baseline', 'none', 'all']

def resolve_ref_tag(ds):
    if REF_TAG is not None:
        return REF_TAG
    tags = list(runs_by_ds[ds])
    for c in _REF_CANDIDATES:
        if c in tags:
            return c
    return None

ref_tags = {ds: resolve_ref_tag(ds) for ds in DATASETS_ORDER}
for ds, t in ref_tags.items():
    n = len(runs_by_ds[ds])
    print(f'{ds:14s} reference arm = {t!r}   ({n} arm(s) on disk)')

_missing = [ds for ds, t in ref_tags.items() if t is None and runs_by_ds[ds]]
if _missing:
    print(f'\nNo reference arm auto-detected for {_missing}. Look at the tag column above '
          f'and set REF_TAG in the config cell to the un-ablated arm, then re-run.')

pancreas       reference arm = 'full'   (8 arm(s) on disk)
lung           reference arm = 'full'   (8 arm(s) on disk)
pbmc-immune    reference arm = 'full'   (8 arm(s) on disk)


## Step 2 -- the paired test

Three deliberate choices, each of which a sceptical reviewer would otherwise ask about:

**1. Batches are matched by name, not by array position.** `modularity_per_batch.csv` is
indexed by batch, so the two arms are joined on the batch label. Positional pairing
would silently misalign if any arm dropped a batch.

**2. Two-sided is the headline p-value.** Choosing the one-sided direction after seeing
which way the numbers moved is post-hoc. The one-sided p is still reported (clearly
labelled, in the pre-specified direction full > ablated) but the two-sided value is the
one to quote.

**3. No Bonferroni; Benjamini-Hochberg instead.** Wilcoxon on n=8 batches has an exact
floor: the smallest achievable two-sided p is 2/2^8 = 0.0078. Bonferroni over 7 ablation
arms multiplies that to 0.055 -- so on pancreas **no arm could ever reach significance,
no matter how large its effect.** That is a property of the correction, not of the data.
BH controls the false-discovery rate without that floor problem. Raw p is reported
alongside so nothing is hidden.

`n_drop` (how many batches got worse when the term was removed) is the most robust column
here -- it is distribution-free and stays readable when n is small.

In [6]:
def read_per_batch(path):
    """modularity_per_batch.csv -> Series indexed by batch label."""
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, index_col=0)
    num = df.select_dtypes(include=[np.number])
    if num.shape[1] == 0:
        return None
    s = num.iloc[:, 0].astype(float)
    s.index = s.index.astype(str)
    return s[~s.index.duplicated()]


def bh_adjust(pvals):
    """Benjamini-Hochberg. NaNs pass through untouched."""
    p = np.asarray(pvals, dtype=float)
    ok = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    if ok.sum() == 0:
        return out
    idx = np.where(ok)[0]
    order = idx[np.argsort(p[idx])]
    m = len(order)
    prev = 1.0
    for rank, i in enumerate(reversed(order), start=1):
        val = min(prev, p[i] * m / (m - rank + 1))
        out[i] = prev = val
    return out


def stars(p):
    if not np.isfinite(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def paired_vs_ref(ref_s, arm_s):
    """Paired Wilcoxon on batches shared by both arms. Returns a dict of stats."""
    a, b = ref_s.align(arm_s, join='inner')
    n = len(a)
    if n == 0:
        return {'n': 0}
    diff = a.values - b.values          # >0 means the full model scored higher
    res = {
        'n': n,
        'ref_mean': float(a.mean()),  'ref_std': float(a.std(ddof=1)) if n > 1 else np.nan,
        'arm_mean': float(b.mean()),  'arm_std': float(b.std(ddof=1)) if n > 1 else np.nan,
        'mean_diff': float(diff.mean()),
        'median_diff': float(np.median(diff)),
        'n_drop': int((diff > 0).sum()),      # batches where removing the term hurt
        'n_rise': int((diff < 0).sum()),
        'n_tie':  int((diff == 0).sum()),
    }
    if n > 1 and np.any(diff != 0):
        try:
            res['p_two_sided'] = float(wilcoxon(a.values, b.values, alternative='two-sided')[1])
            res['p_one_sided_full_gt'] = float(wilcoxon(a.values, b.values, alternative='greater')[1])
        except ValueError:
            pass
    return res

print('helpers ready')

helpers ready


In [7]:
def modularity_paired_table(ds):
    ref_tag = ref_tags[ds]
    runs = runs_by_ds[ds]
    if ref_tag is None or ref_tag not in runs:
        print(f'[{ds}] no reference arm -- skipped')
        return pd.DataFrame()

    ref_s = read_per_batch(os.path.join(runs[ref_tag], 'modularity_per_batch.csv'))
    if ref_s is None:
        print(f'[{ds}] reference arm has no modularity_per_batch.csv -- skipped')
        return pd.DataFrame()

    rows = []
    for tag, d in runs.items():
        if tag == ref_tag:
            continue
        arm_s = read_per_batch(os.path.join(d, 'modularity_per_batch.csv'))
        if arm_s is None:
            print(f'  [{ds}] {tag}: no modularity_per_batch.csv -- skipped')
            continue
        r = paired_vs_ref(ref_s, arm_s)
        if r.get('n', 0) == 0:
            print(f'  [{ds}] {tag}: no batches shared with the reference -- skipped')
            continue
        rows.append({'dataset': DATASET_DISPLAY.get(ds, ds), 'metric': 'modularity',
                     'ref_tag': ref_tag, 'arm': tag, **r})

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if 'p_two_sided' in df:
        df['q_two_sided'] = bh_adjust(df['p_two_sided'])
        df['sig'] = df['q_two_sided'].map(stars)
    return df

print('ready')

ready


## Step 3 -- rare-cell metrics (per batch, so also pairable)

`rare_celltype_purity_table()` returns the raw per-batch arrays alongside the
mean+-std, in the `_batch_rare_*_per_batch` columns. Those are pairable for the same
reason modularity is: "locally rare" is defined from ground-truth cell-type frequency
*within a batch*, so every arm's array covers the same batches in the same order.

This is where the `L_nassoc` claim should live. Its purity effect is not testable
(per-metacell), but its rare-homogeneity effect is -- and unlike purity, it drops on all
three datasets.

`k_tolerance` is set high on purpose. The default 5% would drop the `- usage loss` arm
before testing it: that arm collapses to 63 active prototypes out of 220, which is the
*result*, not a reason to exclude it. K-matching exists to keep different clustering
*methods* comparable; here every arm is the same method.

In [8]:
from interpretable_ssl.evaluation.paper_figures import rare_celltype_purity_table

RARE_METRICS = [
    '_batch_rare_homogeneity_per_batch',
    '_batch_rare_f1_macro_per_batch',
    '_batch_rare_coverage_per_batch',
    '_batch_rare_cross_batch_homog_per_batch',
]


def rare_paired_table(ds):
    runs, ref_tag = runs_by_ds[ds], ref_tags[ds]
    if ref_tag is None or ref_tag not in runs:
        return pd.DataFrame()

    # rare_celltype_purity_table matches runs by keyword against the folder name
    keywords = {os.path.basename(d): tag for tag, d in runs.items()}
    df_rare = rare_celltype_purity_table(ds, model_keywords=keywords, verbose=False)
    if df_rare.empty:
        print(f'[{ds}] rare table empty -- do the arms have umap_cells.csv?')
        return pd.DataFrame()

    present = [m for m in RARE_METRICS if m in df_rare.columns]
    missing = [m for m in RARE_METRICS if m not in df_rare.columns]
    if missing:
        print(f'[{ds}] not in this run of rare_celltype_purity_table, skipped: {missing}')

    rows = []
    for metric in present:
        if (ds, ref_tag) not in df_rare.index:
            continue
        ref_v = df_rare.loc[(ds, ref_tag), metric]
        if not isinstance(ref_v, list) or not ref_v:
            continue
        ref_s = pd.Series(ref_v, index=[str(i) for i in range(len(ref_v))], dtype=float)
        for tag in runs:
            if tag == ref_tag or (ds, tag) not in df_rare.index:
                continue
            v = df_rare.loc[(ds, tag), metric]
            if not isinstance(v, list) or len(v) != len(ref_v):
                continue
            arm_s = pd.Series(v, index=ref_s.index, dtype=float)
            r = paired_vs_ref(ref_s, arm_s)
            rows.append({'dataset': DATASET_DISPLAY.get(ds, ds),
                         'metric': metric.lstrip('_').replace('_per_batch', ''),
                         'ref_tag': ref_tag, 'arm': tag, **r})

    df = pd.DataFrame(rows)
    if df.empty or 'p_two_sided' not in df:
        return df
    df['q_two_sided'] = df.groupby('metric')['p_two_sided'].transform(
        lambda s: pd.Series(bh_adjust(s.values), index=s.index))
    df['sig'] = df['q_two_sided'].map(stars)
    return df

print('ready')

ready


## Step 4 -- run it, one dataset at a time

Pancreas first. Each dataset's result is written to `OUT_DIR` as soon as it finishes, so
a disconnect never costs a completed dataset.

In [9]:
def run_dataset(ds, force=False):
    out_csv = os.path.join(OUT_DIR, f'paired_{ds}.csv')
    if SKIP_IF_EXISTS and not force and os.path.exists(out_csv):
        print(f'[{ds}] loading saved result ({out_csv})')
        return pd.read_csv(out_csv)

    print(f'=== {ds} ===')
    parts = [modularity_paired_table(ds), rare_paired_table(ds)]
    df = pd.concat([p for p in parts if not p.empty], ignore_index=True) \
        if any(not p.empty for p in parts) else pd.DataFrame()
    if not df.empty:
        df.to_csv(out_csv, index=False)
        print(f'[{ds}] saved -> {out_csv}')
    else:
        print(f'[{ds}] nothing to save')
    return df

res_pancreas = run_dataset('pancreas')
res_pancreas

=== pancreas ===
  [fixed_temp|pancreas] resolving run dir ...
  [full|pancreas] resolving run dir ...
  [kmeans_init|pancreas] resolving run dir ...
  [no_community|pancreas] resolving run dir ...
  [no_nassoc|pancreas] resolving run dir ...
  [no_recon|pancreas] resolving run dir ...
  [no_usage|pancreas] resolving run dir ...
  [stopgrad_off|pancreas] resolving run dir ...
  [kmeans_init|pancreas] run dir resolved (0.0s)  [fixed_temp|pancreas] run dir resolved (0.0s)
  [no_usage|pancreas] run dir resolved (0.0s)  [no_nassoc|pancreas] run dir resolved (0.0s)
  [no_community|pancreas] run dir resolved (0.0s)
  [no_recon|pancreas] run dir resolved (0.0s)
  [stopgrad_off|pancreas] run dir resolved (0.0s)

  [full|pancreas] run dir resolved (0.0s)

  [fixed_temp|pancreas] reading umap_cells.csv ...
  [no_recon|pancreas] reading umap_cells.csv ...
  [full|pancreas] reading umap_cells.csv ...
  [no_nassoc|pancreas] reading umap_cells.csv ...
  [no_community|pancreas] reading umap_cells.csv

,dataset,metric,ref_tag,arm,n,ref_mean,ref_std,arm_mean,arm_std,mean_diff,median_diff,n_drop,n_rise,n_tie,p_two_sided,p_one_sided_full_gt,q_two_sided,sig
0,Pancreas,modularity,full,fixed_temp,9,0.615344,0.080137,0.623038,0.079436,-0.007694,-0.011992,3,6,0,0.425781,0.820312,0.496745,ns
1,Pancreas,modularity,full,kmeans_init,9,0.615344,0.080137,0.593752,0.067978,0.021592,0.022930,7,2,0,0.054688,0.027344,0.076563,ns
2,Pancreas,modularity,full,no_community,9,0.615344,0.080137,0.449124,0.043302,0.166220,0.197979,8,1,0,0.007812,0.003906,0.013672,*
3,Pancreas,modularity,full,no_nassoc,9,0.615344,0.080137,0.656395,0.080278,-0.041051,-0.039948,0,9,0,0.003906,1.000000,0.009115,**
4,Pancreas,modularity,full,no_recon,9,0.615344,0.080137,0.613535,0.077583,0.001809,0.008246,5,4,0,1.000000,0.500000,1.000000,ns
5,Pancreas,modularity,full,no_usage,9,0.615344,0.080137,0.663240,0.083012,-0.047896,-0.060021,0,9,0,0.003906,1.000000,0.009115,**
6,Pancreas,modularity,full,stopgrad_off,9,0.615344,0.080137,0.555382,0.070124,0.059962,0.059480,9,0,0,0.003906,0.001953,0.009115,**
7,Pancreas,batch_rare_homogeneity,full,fixed_temp,8,0.631932,0.176194,0.563313,0.169942,0.068619,0.074863,7,1,0,0.109375,0.054688,0.127604,ns
8,Pancreas,batch_rare_homogeneity,full,kmeans_init,8,0.631932,0.176194,0.482674,0.232510,0.149259,0.154293,8,0,0,0.007812,0.003906,0.027344,*
9,Pancreas,batch_rare_homogeneity,full,no_community,8,0.631932,0.176194,0.406344,0.245764,0.225588,0.208772,7,1,0,0.015625,0.007812,0.027344,*


In [10]:
# The two arms nG29 quoted, on the dataset he quoted them from.
if not res_pancreas.empty:
    focus = res_pancreas[
        res_pancreas['arm'].str.contains('stopgrad|stop_grad|nassoc', case=False, regex=True)
    ]
    cols = ['metric', 'arm', 'n', 'ref_mean', 'arm_mean', 'mean_diff',
            'n_drop', 'n_rise', 'p_two_sided', 'p_one_sided_full_gt', 'q_two_sided', 'sig']
    display(focus[[c for c in cols if c in focus.columns]])
else:
    print('no pancreas results')

,metric,arm,n,ref_mean,arm_mean,mean_diff,n_drop,n_rise,p_two_sided,p_one_sided_full_gt,q_two_sided,sig
3,modularity,no_nassoc,9,0.615344,0.656395,-0.041051,0,9,0.003906,1.000000,0.009115,**
6,modularity,stopgrad_off,9,0.615344,0.555382,0.059962,9,0,0.003906,0.001953,0.009115,**
10,batch_rare_homogeneity,no_nassoc,8,0.631932,0.511199,0.120733,7,1,0.054688,0.027344,0.076563,ns
13,batch_rare_homogeneity,stopgrad_off,8,0.631932,0.505770,0.126162,8,0,0.007812,0.003906,0.027344,*
17,batch_rare_f1_macro,no_nassoc,8,0.588520,0.504419,0.084101,6,2,0.109375,0.054688,0.255208,ns
20,batch_rare_f1_macro,stopgrad_off,8,0.588520,0.518742,0.069778,6,2,0.312500,0.156250,0.437500,ns
24,batch_rare_cross_batch_homog,no_nassoc,8,0.252389,0.344740,-0.092351,1,7,0.148438,0.945312,0.519531,ns
27,batch_rare_cross_batch_homog,stopgrad_off,8,0.252389,0.355758,-0.103369,3,5,0.312500,0.875000,0.535937,ns


In [11]:
res_lung = run_dataset('lung')
res_lung

=== lung ===
  [fixed_temp|lung] resolving run dir ...
  [full|lung] resolving run dir ...
  [kmeans_init|lung] resolving run dir ...
  [no_community|lung] resolving run dir ...
  [no_nassoc|lung] resolving run dir ...
  [no_recon|lung] resolving run dir ...
  [no_usage|lung] resolving run dir ...
  [stopgrad_off|lung] resolving run dir ...
  [fixed_temp|lung] run dir resolved (0.0s)
  [fixed_temp|lung] reading umap_cells.csv ...
  [no_community|lung] run dir resolved (0.0s)
  [full|lung] run dir resolved (0.0s)
  [kmeans_init|lung] run dir resolved (0.0s)
  [no_usage|lung] run dir resolved (0.0s)
  [no_nassoc|lung] run dir resolved (0.0s)
  [stopgrad_off|lung] run dir resolved (0.0s)
  [no_recon|lung] run dir resolved (0.0s)
  [kmeans_init|lung] reading umap_cells.csv ...
  [full|lung] reading umap_cells.csv ...
  [no_usage|lung] reading umap_cells.csv ...
  [no_community|lung] reading umap_cells.csv ...
  [stopgrad_off|lung] reading umap_cells.csv ...  [no_recon|lung] reading umap_ce

,dataset,metric,ref_tag,arm,n,ref_mean,ref_std,arm_mean,arm_std,mean_diff,median_diff,n_drop,n_rise,n_tie,p_two_sided,p_one_sided_full_gt,q_two_sided,sig
0,Lung,modularity,full,fixed_temp,16,0.655442,0.030173,0.674063,0.026423,-0.018621,-0.021603,2,14,0,0.001678,0.999344,0.002937,**
1,Lung,modularity,full,kmeans_init,16,0.655442,0.030173,0.673782,0.031126,-0.018340,-0.025589,4,12,0,0.083252,0.963043,0.097127,ns
2,Lung,modularity,full,no_community,16,0.655442,0.030173,0.483392,0.048698,0.172049,0.179881,16,0,0,0.000031,0.000015,0.000071,***
3,Lung,modularity,full,no_nassoc,16,0.655442,0.030173,0.707663,0.032758,-0.052221,-0.048677,0,16,0,0.000031,1.000000,0.000071,***
4,Lung,modularity,full,no_recon,16,0.655442,0.030173,0.666247,0.026747,-0.010805,-0.005488,5,11,0,0.116669,0.947708,0.116669,ns
5,Lung,modularity,full,no_usage,16,0.655442,0.030173,0.697742,0.021829,-0.042300,-0.038423,0,16,0,0.000031,1.000000,0.000071,***
6,Lung,modularity,full,stopgrad_off,16,0.655442,0.030173,0.666704,0.029191,-0.011262,-0.008851,4,12,0,0.018250,0.992249,0.025549,*
7,Lung,batch_rare_homogeneity,full,fixed_temp,15,0.555216,0.240570,0.552837,0.238654,0.002379,-0.000731,7,8,0,0.977966,0.488983,0.977966,ns
8,Lung,batch_rare_homogeneity,full,kmeans_init,15,0.555216,0.240570,0.569049,0.232237,-0.013833,0.005604,9,6,0,0.719727,0.660614,0.937948,ns
9,Lung,batch_rare_homogeneity,full,no_community,15,0.555216,0.240570,0.539878,0.243879,0.015339,0.020023,10,5,0,0.524475,0.262238,0.917831,ns


In [12]:
res_immune = run_dataset('pbmc-immune')
res_immune

=== pbmc-immune ===
  [fixed_temp|pbmc-immune] resolving run dir ...  [full|pbmc-immune] resolving run dir ...
  [kmeans_init|pbmc-immune] resolving run dir ...

  [no_community|pbmc-immune] resolving run dir ...
  [no_nassoc|pbmc-immune] resolving run dir ...
  [no_recon|pbmc-immune] resolving run dir ...
  [no_usage|pbmc-immune] resolving run dir ...
  [stopgrad_off|pbmc-immune] resolving run dir ...
  [kmeans_init|pbmc-immune] run dir resolved (0.0s)  [full|pbmc-immune] run dir resolved (0.0s)
  [no_nassoc|pbmc-immune] run dir resolved (0.0s)
  [no_community|pbmc-immune] run dir resolved (0.0s)
  [no_recon|pbmc-immune] run dir resolved (0.0s)
  [no_usage|pbmc-immune] run dir resolved (0.0s)

  [fixed_temp|pbmc-immune] run dir resolved (0.0s)
  [stopgrad_off|pbmc-immune] run dir resolved (0.0s)  [no_nassoc|pbmc-immune] reading umap_cells.csv ...

  [kmeans_init|pbmc-immune] reading umap_cells.csv ...
  [no_usage|pbmc-immune] reading umap_cells.csv ...
  [full|pbmc-immune] reading uma

,dataset,metric,ref_tag,arm,n,ref_mean,ref_std,arm_mean,arm_std,mean_diff,median_diff,n_drop,n_rise,n_tie,p_two_sided,p_one_sided_full_gt,q_two_sided,sig
0,Immune,modularity,full,fixed_temp,5,0.623416,0.059632,0.645323,0.061833,-0.021907,-0.025080,1,4,0,0.125000,0.96875,0.218750,ns
1,Immune,modularity,full,kmeans_init,5,0.623416,0.059632,0.630999,0.060971,-0.007583,-0.010345,2,3,0,0.312500,0.90625,0.437500,ns
2,Immune,modularity,full,no_community,5,0.623416,0.059632,0.378915,0.091234,0.244501,0.268388,5,0,0,0.062500,0.03125,0.218750,ns
3,Immune,modularity,full,no_nassoc,5,0.623416,0.059632,0.653144,0.056427,-0.029728,-0.031365,0,5,0,0.062500,1.00000,0.218750,ns
4,Immune,modularity,full,no_recon,5,0.623416,0.059632,0.629762,0.071566,-0.006346,0.002753,3,2,0,0.812500,0.68750,0.812500,ns
5,Immune,modularity,full,no_usage,5,0.623416,0.059632,0.646848,0.048371,-0.023432,-0.028888,1,4,0,0.125000,0.96875,0.218750,ns
6,Immune,modularity,full,stopgrad_off,5,0.623416,0.059632,0.605952,0.092774,0.017464,-0.009325,1,4,0,0.625000,0.78125,0.729167,ns
7,Immune,batch_rare_homogeneity,full,fixed_temp,5,0.904687,0.035547,0.852621,0.073236,0.052066,0.024473,5,0,0,0.062500,0.03125,0.109375,ns
8,Immune,batch_rare_homogeneity,full,kmeans_init,5,0.904687,0.035547,0.807985,0.143387,0.096702,0.082796,4,1,0,0.187500,0.09375,0.187500,ns
9,Immune,batch_rare_homogeneity,full,no_community,5,0.904687,0.035547,0.696802,0.132403,0.207885,0.189298,5,0,0,0.062500,0.03125,0.109375,ns


## Step 5 -- combined table + a copy-pasteable summary

`n_drop` reads as "batches where removing this term made the metric worse", out of `n`.

Read the exact-test floor before quoting any p-value: with n batches, the smallest
two-sided p a Wilcoxon can produce is 2/2^n. Pancreas (n=8) bottoms out at 0.0078 and
Immune (n=5) at 0.0625 -- **Immune cannot reach p<0.05 at all**, so an "ns" there means
underpowered, not refuted. That is the same caveat already used for the Harmony/ComBat
tables, so the framing stays consistent.

In [13]:
_all = [d for d in (res_pancreas, res_lung, res_immune) if isinstance(d, pd.DataFrame) and not d.empty]
df_all = pd.concat(_all, ignore_index=True) if _all else pd.DataFrame()

if not df_all.empty:
    df_all.to_csv(os.path.join(OUT_DIR, 'paired_all_datasets.csv'), index=False)
    print('saved ->', os.path.join(OUT_DIR, 'paired_all_datasets.csv'))

    for n in sorted(df_all['n'].unique()):
        print(f'  n={n} batches: smallest possible two-sided p = {2 / 2**n:.4f}')

    cols = ['dataset', 'metric', 'arm', 'n', 'ref_mean', 'arm_mean', 'mean_diff',
            'n_drop', 'p_two_sided', 'q_two_sided', 'sig']
    display(df_all[[c for c in cols if c in df_all.columns]])
else:
    print('nothing to combine')

saved -> /content/drive/MyDrive/rebuttal_results/ablation_paired/paired_all_datasets.csv
  n=5 batches: smallest possible two-sided p = 0.0625
  n=8 batches: smallest possible two-sided p = 0.0078
  n=9 batches: smallest possible two-sided p = 0.0039
  n=15 batches: smallest possible two-sided p = 0.0001
  n=16 batches: smallest possible two-sided p = 0.0000


,dataset,metric,arm,n,ref_mean,arm_mean,mean_diff,n_drop,p_two_sided,q_two_sided,sig
0,Pancreas,modularity,fixed_temp,9,0.615344,0.623038,-0.007694,3,0.425781,0.496745,ns
1,Pancreas,modularity,kmeans_init,9,0.615344,0.593752,0.021592,7,0.054688,0.076563,ns
2,Pancreas,modularity,no_community,9,0.615344,0.449124,0.166220,8,0.007812,0.013672,*
3,Pancreas,modularity,no_nassoc,9,0.615344,0.656395,-0.041051,0,0.003906,0.009115,**
4,Pancreas,modularity,no_recon,9,0.615344,0.613535,0.001809,5,1.000000,1.000000,ns
...,...,...,...,...,...,...,...,...,...,...,...
79,Immune,batch_rare_cross_batch_homog,no_community,5,0.311092,0.114942,0.196150,4,0.125000,0.291667,ns
80,Immune,batch_rare_cross_batch_homog,no_nassoc,5,0.311092,0.248406,0.062686,3,0.312500,0.546875,ns
81,Immune,batch_rare_cross_batch_homog,no_recon,5,0.311092,0.270286,0.040807,2,0.715001,0.715001,ns
82,Immune,batch_rare_cross_batch_homog,no_usage,5,0.311092,0.448885,-0.137793,0,0.062500,0.218750,ns


In [14]:
# Markdown version, ready to paste into experiment-results/ or a reviewer reply.
if not df_all.empty:
    lines = ['# Ablation: paired per-batch significance (no retraining)', '',
             'Paired one-sided/two-sided Wilcoxon signed-rank, full model vs each ablated arm,',
             'batches matched by label. `n_drop` = batches where removing the term made the',
             'metric worse. BH-adjusted q across arms within each (dataset, metric).', '']
    for (ds, metric), g in df_all.groupby(['dataset', 'metric'], sort=False):
        n = int(g['n'].iloc[0])
        lines += [f'## {ds} -- {metric} (n={n} batches, exact floor p={2 / 2**n:.4f})', '',
                  '| Arm | full | ablated | diff | batches worse | p (2-sided) | q (BH) | |',
                  '|---|---|---|---|---|---|---|---|']
        for _, r in g.sort_values('mean_diff', ascending=False).iterrows():
            p = r.get('p_two_sided', float('nan'))
            q = r.get('q_two_sided', float('nan'))
            lines.append(
                f"| {r['arm']} | {r['ref_mean']:.3f} | {r['arm_mean']:.3f} | "
                f"{r['mean_diff']:+.3f} | {int(r['n_drop'])}/{n} | "
                f"{p:.4f} | {q:.4f} | {r.get('sig', '')} |"
                if np.isfinite(p) else
                f"| {r['arm']} | {r['ref_mean']:.3f} | {r['arm_mean']:.3f} | "
                f"{r['mean_diff']:+.3f} | {int(r['n_drop'])}/{n} | - | - | |")
        lines.append('')

    md = '\n'.join(lines)
    with open(os.path.join(OUT_DIR, 'paired_significance_summary.md'), 'w') as f:
        f.write(md)
    print('saved ->', os.path.join(OUT_DIR, 'paired_significance_summary.md'))
    print()
    print(md)

saved -> /content/drive/MyDrive/rebuttal_results/ablation_paired/paired_significance_summary.md

# Ablation: paired per-batch significance (no retraining)

Paired one-sided/two-sided Wilcoxon signed-rank, full model vs each ablated arm,
batches matched by label. `n_drop` = batches where removing the term made the
metric worse. BH-adjusted q across arms within each (dataset, metric).

## Pancreas -- modularity (n=9 batches, exact floor p=0.0039)

| Arm | full | ablated | diff | batches worse | p (2-sided) | q (BH) | |
|---|---|---|---|---|---|---|---|
| no_community | 0.615 | 0.449 | +0.166 | 8/9 | 0.0078 | 0.0137 | * |
| stopgrad_off | 0.615 | 0.555 | +0.060 | 9/9 | 0.0039 | 0.0091 | ** |
| kmeans_init | 0.615 | 0.594 | +0.022 | 7/9 | 0.0547 | 0.0766 | ns |
| no_recon | 0.615 | 0.614 | +0.002 | 5/9 | 1.0000 | 1.0000 | ns |
| fixed_temp | 0.615 | 0.623 | -0.008 | 3/9 | 0.4258 | 0.4967 | ns |
| no_nassoc | 0.615 | 0.656 | -0.041 | 0/9 | 0.0039 | 0.0091 | ** |
| no_usage | 0.615 | 0.663 |

## Optional cross-check against the codebase's own function

`graph_batch_significance_paired()` computes the same test for the baseline tables.
Running it here on the ablation arms should reproduce the modularity `n_wins` and raw
one-sided p above. `k_tolerance` is disabled (`10.0`) for the reason given in Step 3 --
left at its 0.05 default it would drop the `- usage loss` arm.

A mismatch means the discovery step resolved a different folder than
`_resolve_run_dir()` did, not that the statistics disagree.

In [15]:
from interpretable_ssl.evaluation.paper_figures import graph_batch_significance_paired

ds = 'pancreas'
kw = {os.path.basename(d): tag for tag, d in runs_by_ds[ds].items()}
try:
    chk = graph_batch_significance_paired(
        [ds], kw, ref_name=ref_tags[ds],
        dataset_display_names=DATASET_DISPLAY, k_tolerance=10.0,
    )
    display(chk[[c for c in ['method', 'n', 'mean', 'n_wins', 'p_vs_ref', 'p_adj', 'sig']
                 if c in chk.columns]])
except Exception as e:
    print(f'cross-check unavailable ({type(e).__name__}: {e}) -- '
          f'the Step 2 result stands on its own, this cell is only a redundancy check')

,method,n,mean,n_wins,p_vs_ref,p_adj,sig
0,fixed_temp,9,0.623038,3.0,0.820312,1.000000,ns
1,full,9,0.615344,NaN,NaN,NaN,NaN
2,kmeans_init,9,0.593752,7.0,0.027344,0.191406,ns
3,no_community,9,0.449124,8.0,0.003906,0.027344,*
4,no_nassoc,9,0.656395,0.0,1.000000,1.000000,ns
5,no_recon,9,0.613535,5.0,0.500000,1.000000,ns
6,no_usage,9,0.663240,0.0,1.000000,1.000000,ns
7,stopgrad_off,9,0.555382,9.0,0.001953,0.013672,*
